In [1]:
from transformers import AutoImageProcessor
import torch.optim as optim
import webdataset as wds
from pathlib import Path
from modules import *
import csv

In [5]:
#Constants
DATASET_PATH = "/home/austen/GeoDataset/dataset_sharded"
BATCH_SIZE = 4
WORKERS = 4
S2_LEVELS = range(3, 7)
S2_LEVEL_WEIGHTS = [0.4, 0.6, 0.8, 1.0]
LEARNING_RATE = 8e-5
PRETRAINED_MODEL_ID = "facebook/convnext-base-384" #"facebook/convnext-base-384" #
EPOCHS = 12
CHECKPOINT_PATH = "checkpoints/checkpoint_large_3.pt"
DEVICE = "cuda"
LOGFILE = "training_log.csv"
MULTI_GPU = True
RESIZE = 0 #384

In [6]:
#Setup
processor = AutoImageProcessor.from_pretrained(PRETRAINED_MODEL_ID, use_fast=True)
if RESIZE > 0:
    processor.do_resize = True
    processor.size = {"shortest_edge": RESIZE}
else:
    processor.do_resize = False


dataset = GeoWebDataset(
    DATASET_PATH, 
    processor, 
    levels=S2_LEVELS, 
    shuffle=False,
    num_shards_limit = None
)

model = HierarchicalConvNeXt(
    pretrained_name = PRETRAINED_MODEL_ID,
    num_classes = dataset.num_classes_list[-1],
    freeze = False
)

parent_table = build_parent_tables(Path(DATASET_PATH) / "s2_labels", S2_LEVELS)
hier_loss = HierarchicalLoss(
    levels=S2_LEVELS,
    parents=parent_table,
    weights=S2_LEVEL_WEIGHTS,
    num_classes_per_level=dataset.num_classes_list,
).to(DEVICE)

loader = (
    wds.WebLoader(
        dataset.dataset,
        num_workers=WORKERS,
        batch_size=BATCH_SIZE,
        pin_memory=True,
        prefetch_factor=2,      
        persistent_workers=True
    )
)

trainer = Trainer(model, loader, hier_loss, DEVICE, MULTI_GPU)

optimizer = optim.AdamW(
    model.parameters(), 
    lr=LEARNING_RATE,
    fused=True #might be faster
)

#Create log file
if not Path(LOGFILE).exists():
    with open(LOGFILE, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["epoch", "loss"])

Using 2 GPUs


In [7]:
#Training
start_epoch = trainer.load_checkpoint(optimizer, CHECKPOINT_PATH)

for epoch in range(start_epoch, EPOCHS):
    print(f"\nEpoch {epoch}")
    
    avg_loss = trainer.train_epoch(optimizer, weights=S2_LEVEL_WEIGHTS)
    print(f"Average loss: {avg_loss:.4f}")

    with open(LOGFILE, "a", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([epoch, avg_loss])

    trainer.save_checkpoint(optimizer, epoch, CHECKPOINT_PATH)

Loaded checkpoint from epoch 1

Epoch 2


62it [00:17,  3.47it/s]


KeyboardInterrupt: 

In [ ]:
#Evaluation
start_epoch = trainer.load_checkpoint(optimizer, CHECKPOINT_PATH)

evaluator = Evaluator(
    trainer.model,
    loader,
    levels=S2_LEVELS,
    parents=parent_table,
    device=DEVICE
)
metrics = evaluator.evaluate(max_batches=100)
print(metrics)

In [ ]:
print("Random Guessing accuracy")
for x in dataset.num_classes_list:
    print(1 / x)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("training_log.csv")

plt.plot(df["epoch"], df["loss"])
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training Loss")
plt.grid(True)
plt.show()